# 01 数据、标签与80项候选特征

本节属于课程进阶模块。代码、教学构造、历史实测、待执行实验分别标注。云执行默认关闭；真实金融数据通过仓库外的私有快照接入。原始行情、逐样本标签、角度、核矩阵、参数与预测不得存入公开Notebook输出。

**来源**：[L001 ICLR正式版](https://proceedings.iclr.cc/paper_files/paper/2024/file/4d9944ab3330fe6af8efb9260aa9f307-Paper-Conference.pdf)、[本课程协议](../../docs/天衍云端复现协议.md)。署名：Julius Zhou、AUFE、Cong Lei。代码MIT，原创教学文字与图表CC BY 4.0。


In [1]:
from pathlib import Path
import sys, os, json
import numpy as np
import pandas as pd
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "课程大纲.md").exists())
MODULE = COURSE / "experiments/qukernet_finance"
sys.path.insert(0, str(MODULE))
PUBLIC = COURSE / "reports/qukernet_cloud_v2"
ENABLE_CLOUD = False
print("默认云执行关闭；当前Notebook不读取凭据、不提交任务。")


默认云执行关闭；当前Notebook不读取凭据、不提交任务。


## 金融问题与时间轴
收盘日$t$生成信号，标签$y_t=O_{t+6}/O_{t+1}-1$到$t+6$开盘才成熟。分类取$1[y_t>0]$，回归保留连续值。把“样本日期在训练期”误当“标签已经可得”，会造成时间泄漏。

本节输入是字段字典与私有冻结快照；输出为训练/验证/季度评价数组及审计清单。训练期选择40项，验证与评价复用同一变换。预热/验证不能回流训练mRMR。

In [2]:
from cloud_v2.features80 import feature_dictionary
dictionary=pd.DataFrame(feature_dictionary())
assert len(dictionary)==80 and dictionary.id.is_unique
display(dictionary[["id","name","formula","window_sessions"]].head(12))
print("最长回看:", dictionary.window_sessions.max(), "个市场交易日")

,id,name,formula,window_sessions
0,F01,log_momentum_1,log(C[t]/C[t-1]),1
1,F02,log_momentum_2,log(C[t]/C[t-2]),2
2,F03,log_momentum_3,log(C[t]/C[t-3]),3
3,F04,log_momentum_5,log(C[t]/C[t-5]),5
4,F05,log_momentum_10,log(C[t]/C[t-10]),10
5,F06,log_momentum_15,log(C[t]/C[t-15]),15
6,F07,log_momentum_20,log(C[t]/C[t-20]),20
7,F08,log_momentum_40,log(C[t]/C[t-40]),40
8,F09,log_momentum_60,log(C[t]/C[t-60]),60
9,F10,log_momentum_120,log(C[t]/C[t-120]),120


最长回看: 120 个市场交易日


## 逐项处理而非统一填零
停牌、缺失日与真实零量不同；日历不压缩。H=L时形态比例按已声明规则为0；零方差相关系数、退化偏度等记缺失。mRMR先排除训练内常数和完全重复列。历史实际准备中F52零成交量比例为常数，80项定义只有79项可选；这不是80个独立信号。

缩放为$\pi\,\mathrm{clip}((x-\mathrm{median})/\mathrm{IQR},-3,3)/3$。仅IQR确为0时按常数处理，不能用固定绝对阈值删掉小单位的流动性指标。

In [3]:
from cloud_v2.data80 import fit_scaler, apply_scaler
# 解析构造：单位改变不应改变缩放后信息。
x=np.array([[1.0,1e-10],[2.0,2e-10],[4.0,4e-10],[8.0,8e-10]])
z=apply_scaler(x,fit_scaler(x))
assert np.allclose(z[:,0],z[:,1])
print("不同量纲的同一变化关系得到一致角度。")

不同量纲的同一变化关系得到一致角度。


## 使用私有真实数据
在命令行调用`python -m cloud_v2.data80 --input PANEL --calendar CALENDAR --output PRIVATE_OUTPUT --quarter 2024Q1 --size 32 --task classification`。所有路径由学习者自己的私有配置提供，不写入本Notebook。模块拒绝网络输出和课程内私有结果。

**验收与作业**：说明为何先按标签成熟净化日历再分日期箱；给出一例“随机打乱时间能提高验证分数但不可信”的机制。检查训练/验证每条标签结束时间、预处理哈希、输入哈希与可选特征数。数据不足或类别退化要形成失败记录。